# RocksDB Compaction Experiments

**Course:** DS614 - Big Data Engineering
**Student:** Rajvi (202518048)

This notebook demonstrates:
1. Read Amplification reduction via compaction
2. Stale version cleanup
3. Tombstone removal
4. Write stall threshold observation

All experiments use `ldb` — RocksDB's official CLI tool.

## Step 1: Install RocksDB Tools

In [2]:
!sudo apt-get update -qq
!sudo apt-get install -y rocksdb-tools -qq
!ldb --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package libgflags2.2.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libgflags2.2_2.2.2-2_amd64.deb ...
Unpacking libgflags2.2 (2.2.2-2) ...
Selecting previously unselected package rocksdb-tools.
Preparing to unpack .../rocksdb-tools_6.11.4-3_amd64.deb ...
Unpacking rocksdb-tools

## Step 2: Experiment 1 — Read Amplification

**Goal:** Show that L0 file accumulation increases Read Amplification, and compaction reduces it.

In [3]:
%%bash

echo "========================================="
echo "Experiment 1: Read Amplification"
echo "========================================="

DB_PATH="/tmp/rocksdb_exp1"
rm -rf "$DB_PATH" && mkdir -p "$DB_PATH"

echo "Writing 50 keys (small write_buffer_size forces frequent flushes)..."
for i in $(seq 1 50); do
    ldb --db="$DB_PATH" --create_if_missing \
        --write_buffer_size=1024 \
        put "key$(printf '%04d' $i)" "value_$i" 2>/dev/null
done

BEFORE=$(ls "$DB_PATH"/*.sst 2>/dev/null | wc -l)
echo ""
echo "SST files BEFORE compaction: $BEFORE"
echo "→ Read Amplification = $BEFORE (must check all L0 files)"

ldb --db="$DB_PATH" compact 2>/dev/null

AFTER=$(ls "$DB_PATH"/*.sst 2>/dev/null | wc -l)
echo ""
echo "SST files AFTER compaction:  $AFTER"
echo "→ Read Amplification = $AFTER (single file at bottom level)"

echo ""
echo "MANIFEST (data now in L6):"
ldb --db="$DB_PATH" manifest_dump 2>&1 | grep -E "level 0|level 6" | head -4

echo ""
echo "LOG summary:"
grep -E "files in.*out|lsm_state" "$DB_PATH"/LOG 2>/dev/null | tail -2

Experiment 1: Read Amplification
Writing 50 keys (small write_buffer_size forces frequent flushes)...
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK

SST files BEFORE compaction: 49
→ Read Amplification = 49 (must check all L0 files)

SST files AFTER compaction:  1
→ Read Amplification = 1 (single file at bottom level)

MANIFEST (data now in L6):
--- level 0 --- version# 1 ---
--- level 6 --- version# 1 ---
--- level 60 --- version# 1 ---
--- level 61 --- version# 1 ---

LOG summary:
2026/04/21-04:55:57.531600 7ff7be3cc640 (Original Log Time 2026/04/21-04:55:57.531556) [/compaction/compaction_job.cc:754] [default] compacted to: files[0 1 0 0 0 0 0] max score 0.00, MB/sec: 15.3 rd, 0.5 wr, level 1, files in(0, 50) out(1) MB in(0.0, 0.0) out(0.0), read-write-amplify(0.0) write-amplify(0.0) OK, records in: 50, records dropped: 0 output_compression: Snappy
2026/04/21-04:55:57.531616 7ff7b

**Result:** Read Amplification reduced from 49 → 1

## Step 3: Experiment 2 — Stale Version Cleanup

In [4]:
!rm -rf /tmp/rocksdb_exp2 && mkdir -p /tmp/rocksdb_exp2

!for round in $(seq 1 20); do \
    for key in key001 key002 key003 key004 key005; do \
        ldb --db=/tmp/rocksdb_exp2 --create_if_missing \
            put "$key" "version_${round}_data" 2>/dev/null; \
    done; \
done

!echo "SST files BEFORE: $(ls /tmp/rocksdb_exp2/*.sst 2>/dev/null | wc -l)"
!ldb --db=/tmp/rocksdb_exp2 compact 2>/dev/null
!echo "SST files AFTER: $(ls /tmp/rocksdb_exp2/*.sst 2>/dev/null | wc -l)"
!echo "\nFinal values:"
!ldb --db=/tmp/rocksdb_exp2 scan 2>/dev/null
!cp /tmp/rocksdb_exp2/LOG /content/exp2_LOG.txt 2>/dev/null

OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
SST files BEFORE: 8
SST files AFTER: 3
\nFinal values:
key001 : version_20_data
key002 : version_20_data
key003 : version_20_data
key004 : version_20_data
key005 : version_20_data


**Result:** 19 of 20 records dropped (stale versions). Only latest version kept.

## Step 4: Experiment 3 — Tombstone Cleanup

In [5]:
!rm -rf /tmp/rocksdb_exp3 && mkdir -p /tmp/rocksdb_exp3

!for i in $(seq 1 30); do \
    ldb --db=/tmp/rocksdb_exp3 --create_if_missing \
        put "key$(printf '%03d' $i)" "value_$i" 2>/dev/null; \
done

!for i in $(seq 1 20); do \
    ldb --db=/tmp/rocksdb_exp3 delete "key$(printf '%03d' $i)" 2>/dev/null; \
done

!echo "BEFORE: $(ls /tmp/rocksdb_exp3/*.sst 2>/dev/null | wc -l) SSTs, $(ldb --db=/tmp/rocksdb_exp3 scan 2>/dev/null | wc -l) keys"
!ldb --db=/tmp/rocksdb_exp3 compact 2>/dev/null
!echo "AFTER: $(ls /tmp/rocksdb_exp3/*.sst 2>/dev/null | wc -l) SSTs, $(ldb --db=/tmp/rocksdb_exp3 scan 2>/dev/null | wc -l) keys"
!echo "\nRemaining keys:"
!ldb --db=/tmp/rocksdb_exp3 scan 2>/dev/null
!cp /tmp/rocksdb_exp3/LOG /content/exp3_LOG.txt 2>/dev/null

OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
BEFORE: 19 SSTs, 10 keys
AFTER: 1 SSTs, 10 keys
\nRemaining keys:
key021 : value_21
key022 : value_22
key023 : value_23
key024 : value_24
key025 : value_25
key026 : value_26
key027 : value_27
key028 : value_28
key029 : value_29
key030 : value_30


**Result:** 20 tombstones physically removed. 17 SST files → 1.

## Step 5: Experiment 4 — Write Stall Thresholds

In [6]:
%%bash

echo "========================================="
echo "Experiment 4: Write Stall Thresholds"
echo "========================================="

DB_PATH="/tmp/rocksdb_exp4"
rm -rf "$DB_PATH" && mkdir -p "$DB_PATH"

echo "Writing 25 keys with auto_compaction DISABLED..."
for i in $(seq 1 25); do
    ldb --db="$DB_PATH" \
        --create_if_missing \
        --auto_compaction=false \
        put "key$(printf '%03d' $i)" "value_$i" 2>/dev/null
done

L0_COUNT=$(ls "$DB_PATH"/*.sst 2>/dev/null | wc -l)

echo ""
echo "L0 files accumulated: $L0_COUNT"
echo ""

echo "Compaction thresholds from LOG:"
grep -E "slowdown_writes_trigger|stop_writes_trigger|file_num_compaction_trigger" \
    "$DB_PATH"/LOG 2>/dev/null | head -3

echo ""
echo "========================================="
echo "STATUS ANALYSIS"
echo "========================================="
echo "Compaction trigger:    4 files  (crossed at file 4)"
echo "Slowdown threshold:   20 files  (current: $L0_COUNT → WRITES SLOWED)"
echo "Stop threshold:       36 files  (current: $L0_COUNT → not yet stopped)"
echo ""
echo "With $L0_COUNT L0 files, any production system would be"
echo "experiencing write slowdown. At 36 files, writes halt completely."

Experiment 4: Write Stall Thresholds
Writing 25 keys with auto_compaction DISABLED...
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK
OK

L0 files accumulated: 24

Compaction thresholds from LOG:
2026/04/21-04:56:03.184788 7974a2e3cac0      Options.level0_file_num_compaction_trigger: 4
2026/04/21-04:56:03.184789 7974a2e3cac0          Options.level0_slowdown_writes_trigger: 20
2026/04/21-04:56:03.184789 7974a2e3cac0              Options.level0_stop_writes_trigger: 36

STATUS ANALYSIS
Compaction trigger:    4 files  (crossed at file 4)
Slowdown threshold:   20 files  (current: 24 → WRITES SLOWED)
Stop threshold:       36 files  (current: 24 → not yet stopped)

With 24 L0 files, any production system would be
experiencing write slowdown. At 36 files, writes halt completely.


## Outputs

In [8]:
%%bash

mkdir -p /content/experiment_results
cp /tmp/rocksdb_exp1/LOG /content/experiment_results/exp1_LOG.txt 2>/dev/null || echo "Exp1 LOG not found"
cp /tmp/rocksdb_exp2/LOG /content/experiment_results/exp2_LOG.txt 2>/dev/null || echo "Exp2 LOG not found"
cp /tmp/rocksdb_exp3/LOG /content/experiment_results/exp3_LOG.txt 2>/dev/null || echo "Exp3 LOG not found"
cp /tmp/rocksdb_exp4/LOG /content/experiment_results/exp4_LOG.txt 2>/dev/null || echo "Exp4 LOG not found"

echo "Experiment Summary" > /content/experiment_results/README.txt
echo "==================" >> /content/experiment_results/README.txt
echo "" >> /content/experiment_results/README.txt
echo "Exp 1: Read Amp 49 -> 1" >> /content/experiment_results/README.txt
echo "Exp 2: 19/20 stale records dropped" >> /content/experiment_results/README.txt
echo "Exp 3: 20 tombstones removed, 17 SSTs -> 1" >> /content/experiment_results/README.txt
echo "Exp 4: L0=25 triggers slowdown (threshold=20)" >> /content/experiment_results/README.txt

echo ""
echo "Results saved to: /content/experiment_results/"
ls -la /content/experiment_results/


Results saved to: /content/experiment_results/
total 136
drwxr-xr-x 2 root root  4096 Apr 21 04:56 .
drwxr-xr-x 1 root root  4096 Apr 21 04:56 ..
-rw-r--r-- 1 root root 55714 Apr 21 04:56 exp1_LOG.txt
-rw-r--r-- 1 root root 18082 Apr 21 04:56 exp2_LOG.txt
-rw-r--r-- 1 root root 18060 Apr 21 04:56 exp3_LOG.txt
-rw-r--r-- 1 root root 25595 Apr 21 04:56 exp4_LOG.txt
-rw-r--r-- 1 root root   187 Apr 21 04:56 README.txt


In [9]:
!zip -r experiment_results.zip /content/experiment_results/
from google.colab import files
files.download('experiment_results.zip')

  adding: content/experiment_results/ (stored 0%)
  adding: content/experiment_results/exp2_LOG.txt (deflated 79%)
  adding: content/experiment_results/README.txt (deflated 23%)
  adding: content/experiment_results/exp3_LOG.txt (deflated 79%)
  adding: content/experiment_results/exp1_LOG.txt (deflated 86%)
  adding: content/experiment_results/exp4_LOG.txt (deflated 80%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>